[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/05_finetune_and_compare.ipynb)

# Step 5 — Fine-Tune and Compare

LoRA fine-tune a small model on two synthetic corpora, then re-run the Step 1 test set.

## Learning objectives
- Prepare an instruction dataset for SFT
- Fine-tune with TRL + PEFT (4-bit LoRA)
- Compare a same-stack HF 4-bit control vs SFT on quality-filtered Q&A vs SFT on instruction back-translation, including failure modes

## Prerequisites
1. Generated `synthetic_filtered.jsonl` (notebook 03), `synthetic_back_translation.jsonl` (notebook 04), and `test_set.jsonl` (notebook 01)
2. Install SFT dependencies:
    ``` bash
    uv sync --group text-sft
    ```
3. An NVIDIA GPU with CUDA (LoRA is 4-bit bitsandbytes; not via Ollama)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    BASELINE_PREDICTIONS_PATH,
    BASELINE_SCORES_PATH,
    COMPARISON_REPORT_PATH,
    DEFAULT_EVAL_MAX_TOKENS,
    FAILURE_MODE_GUIDANCE,
    FINETUNED_PREDICTIONS_PATH,
    HF_BASE_PREDICTIONS_PATH,
    HF_BASE_SCORES_PATH,
    RESULTS_DIR,
    SYNTHETIC_FILTERED_PATH,
    SYNTHETIC_IBT_PATH,
    TEST_SET_PATH,
    FailureMode,
    Hf4BitInferenceClient,
    QASample,
    build_sft_dataset,
    compare_summaries,
    create_judge_client,
    load_implementation_dotenv,
    load_typed_jsonl,
    qa_samples_to_messages,
    read_json,
    read_jsonl,
    run_inference,
    save_baseline_results,
    score_predictions,
    train_lora_sft,
    use_repo_root,
    write_json,
)
from aieng.syn_data.text.sft import PeftInferenceClient
from rich.console import Console
from rich.table import Table


load_implementation_dotenv()
ROOT = use_repo_root(Path("."))

MODELS_DIR = ROOT / "implementations" / "qa_text_generation" / "models"
IBT_ADAPTER_DIR = MODELS_DIR / "lora_adapter"
FILTERED_ADAPTER_DIR = MODELS_DIR / "lora_adapter_filtered"
FINETUNED_FILTERED_PREDICTIONS_PATH = RESULTS_DIR / "finetuned_filtered_predictions.jsonl"
FINETUNED_FILTERED_SCORES_PATH = RESULTS_DIR / "finetuned_filtered_scores.json"
BASE_MODEL = os.getenv("SFT_BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
EVAL_MAX_TOKENS = DEFAULT_EVAL_MAX_TOKENS

console = Console(width=140)

## 1. Load training and test data

Load the quality-filtered corpus from notebook 03 and the instruction-back-translation (IBT) corpus from notebook 04. Both are scored later on the same Step 1 test set.

In [4]:
ibt_train_samples = load_typed_jsonl(SYNTHETIC_IBT_PATH, QASample.from_dict)
filtered_train_samples = load_typed_jsonl(SYNTHETIC_FILTERED_PATH, QASample.from_dict)
test_samples = load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)
print(
    f"IBT train: {len(ibt_train_samples)} | filtered train: {len(filtered_train_samples)} | test: {len(test_samples)}"
)

IBT train: 36 | filtered train: 160 | test: 56


## 2. Same-stack control: Hugging Face 4-bit (no adapter)

Notebook 01 scored **Ollama** (`qwen2.5:0.5b-instruct` GGUF). LoRA eval uses **Hugging Face** `Qwen/Qwen2.5-0.5B-Instruct` in bitsandbytes nf4. Those are different engines, quantizations, so Ollama vs LoRA deltas are not attributable to SFT.

This cell scores the **un-adapted HF 4-bit base** with the same `generate()` path and `max_tokens` as the adapters. Later tables use this as the control. Notebook 01 Ollama scores are kept only as a side-by-side stack check.

Unload this client before LoRA training so it does not hold GPU memory.

In [ ]:
judge = create_judge_client()
ollama_report = read_json(BASELINE_SCORES_PATH) if BASELINE_SCORES_PATH.exists() else None

hf_base_client = Hf4BitInferenceClient(BASE_MODEL)
hf_base_predictions = run_inference(hf_base_client, test_samples, max_tokens=EVAL_MAX_TOKENS)
hf_base_scores = score_predictions(judge, test_samples, hf_base_predictions)
hf_base_summary = save_baseline_results(
    hf_base_predictions,
    hf_base_scores,
    test_samples,
    predictions_path=HF_BASE_PREDICTIONS_PATH,
    scores_path=HF_BASE_SCORES_PATH,
)
hf_base_client.release()
control_label = "HF 4-bit"
console.print(f"[bold green]Same-stack control saved ({control_label}, max_tokens={EVAL_MAX_TOKENS}).[/bold green]")


table = Table(title=f"{control_label} control evaluation")
table.add_column("Metric", justify="left", style="cyan", no_wrap=True)
table.add_column(control_label, justify="right", style="yellow")
if ollama_report:
    table.add_column("Ollama (nb 01)", justify="right", style="dim")
    table.add_column("Stack gap", justify="right", style="magenta")
for metric, score in hf_base_summary.get("overall", {}).items():
    row = [metric, f"{score:.3f}"]
    if ollama_report:
        ollama_val = ollama_report["overall"].get(metric)
        delta = compare_summaries(ollama_report["overall"], hf_base_summary["overall"]).get(metric)
        row.append(f"{ollama_val:.3f}" if isinstance(ollama_val, (int, float)) else "—")
        row.append(f"{delta:+.3f}" if isinstance(delta, (int, float)) else "—")
    table.add_row(*row)
console.print(table)

Same-stack control saved (HF 4-bit, max_tokens=512).

                   HF 4-bit control evaluation                   
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric                ┃ HF 4-bit ┃ Ollama (nb 01) ┃ Stack gap ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ correctness           │    3.348 │          3.911 │    -0.562 │
│ coherence             │    4.705 │          4.750 │    -0.045 │
│ instruction_following │    4.821 │          4.750 │    +0.071 │
│ factual_plausibility  │    3.643 │          4.161 │    -0.518 │
│ average               │    4.129 │          4.393 │    -0.263 │
└───────────────────────┴──────────┴────────────────┴───────────┘

## 3. LoRA SFT with TRL (instruction back-translation)

Fine-tune on the grounded IBT corpus. Suggested base model: `Qwen/Qwen2.5-0.5B-Instruct` with 4-bit quantization. The adapter is saved separately from the filtered-corpus run in section 6.


In [15]:
sft_dataset = build_sft_dataset(ibt_train_samples)
print(sft_dataset)
console.print("[bold]Example IBT training row:[/bold]")
qa_samples_to_messages(ibt_train_samples[:1])[0]

ibt_adapter_path = train_lora_sft(
    ibt_train_samples,
    IBT_ADAPTER_DIR,
    base_model=BASE_MODEL,
    num_train_epochs=1.0,
)
console.print(f"[bold green]IBT LoRA adapter saved to {ibt_adapter_path}[/bold green]")

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 36
})


Example IBT training row:

Truncating train dataset: 100%|██████████| 36/36 [00:00<00:00, 3409.70 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 36/36 [00:00<00:00, 8316.07 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


IBT LoRA adapter saved to /home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/models/lora_adapter

## 4. Re-evaluate the IBT adapter on the held-out test set

Uses the same HF 4-bit generate path and `EVAL_MAX_TOKENS` as the section 2 control.


In [16]:
ibt_client = PeftInferenceClient(IBT_ADAPTER_DIR, BASE_MODEL)
ibt_predictions = run_inference(ibt_client, test_samples, max_tokens=EVAL_MAX_TOKENS)
ibt_scores = score_predictions(judge, test_samples, ibt_predictions)

ibt_eval_label = "IBT SFT"
ibt_summary = save_baseline_results(
    ibt_predictions,
    ibt_scores,
    test_samples,
    predictions_path=FINETUNED_PREDICTIONS_PATH,
    scores_path=RESULTS_DIR / "finetuned_scores.json",
)

table = Table(title=f"{ibt_eval_label} Model Evaluation Summary")
table.add_column("Metric", justify="left", style="cyan", no_wrap=True)
table.add_column("Score", justify="right", style="magenta")

for metric, score in ibt_summary["overall"].items():
    table.add_row(metric, f"{score:.3f}")

console.print(table)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 700.81it/s]
2026-08-27 00:34:24,839 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-0 (answer length: 993)
2026-08-27 00:34:26,575 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-1 (answer length: 2315)
2026-08-27 00:34:28,121 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-2 (answer length: 993)
2026-08-27 00:34:29,354 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0009-0 (answer length: 482)
2026-08-27 00:34:30,600 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0009-1 (answer length: 482)
2026-08-27 00:34:31,727 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0009-2 (answer length: 482)
2026-08-27 00:34:33,339 I

IBT SFT Model Evaluation Summary 
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Metric                ┃ Score ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ correctness           │ 4.527 │
│ coherence             │ 4.661 │
│ instruction_following │ 4.643 │
│ factual_plausibility  │ 4.964 │
│ average               │ 4.699 │
└───────────────────────┴───────┘

## 5. Compare same-stack control vs IBT SFT

Deltas are vs the HF 4-bit base (section 2), not vs Ollama.


In [17]:
control_overall = hf_base_summary.get("overall", {})
comparison = {
    "baseline": control_overall,
    "baseline_stack": "hf_4bit",
    "control_label": control_label,
    "ollama_baseline": (ollama_report or {}).get("overall", {}),
    "finetuned": ibt_summary["overall"],
    "finetuned_ibt": ibt_summary["overall"],
    "delta": compare_summaries(control_overall, ibt_summary["overall"]),
    "delta_ibt": compare_summaries(control_overall, ibt_summary["overall"]),
    "by_failure_mode": {
        "baseline": hf_base_summary.get("by_failure_mode", {}),
        "ollama": (ollama_report or {}).get("by_failure_mode", {}),
        "finetuned": ibt_summary.get("by_failure_mode", {}),
        "finetuned_ibt": ibt_summary.get("by_failure_mode", {}),
    },
    "comparison_label": ibt_eval_label,
}
write_json(COMPARISON_REPORT_PATH, comparison)


def _fmt_metric(value, *, signed: bool = False) -> str:
    if not isinstance(value, (int, float)):
        return "—"
    return f"{value:+.3f}" if signed else f"{value:.3f}"


def print_comparison_table(comparison):
    """Print same-stack control vs IBT SFT metrics."""
    control = comparison.get("control_label", "HF 4-bit")
    second_label = comparison.get("comparison_label", "IBT SFT")
    table = Table(title=f"{control} vs IBT SFT (same stack)")
    table.add_column("Metric", justify="left", style="cyan", no_wrap=True)
    table.add_column(control, justify="right", style="yellow")
    table.add_column(second_label, justify="right", style="green")
    table.add_column("Delta", justify="right", style="magenta")

    for metric in comparison["baseline"]:
        table.add_row(
            metric,
            _fmt_metric(comparison["baseline"].get(metric)),
            _fmt_metric(comparison["finetuned_ibt"].get(metric)),
            _fmt_metric(comparison["delta_ibt"].get(metric), signed=True),
        )

    console.print(table)


print_comparison_table(comparison)

           HF 4-bit vs IBT SFT (same stack)            
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓
┃ Metric                ┃ HF 4-bit ┃ IBT SFT ┃  Delta ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━┩
│ correctness           │    3.027 │   4.527 │ +1.500 │
│ coherence             │    4.554 │   4.661 │ +0.107 │
│ instruction_following │    4.518 │   4.643 │ +0.125 │
│ factual_plausibility  │    3.295 │   4.964 │ +1.670 │
│ average               │    3.848 │   4.699 │ +0.850 │
└───────────────────────┴──────────┴─────────┴────────┘

## 6. LoRA SFT on the quality-filtered corpus

Repeat the same LoRA recipe on `synthetic_filtered.jsonl` (notebook 03), **without** instruction back-translation. The adapter is stored in a separate directory so the IBT weights are not overwritten. Same hyperparameters as section 3 so the comparison is about the training data, not the recipe.

This cell unloads the IBT inference client first to free GPU memory.


In [18]:
filtered_sft_dataset = build_sft_dataset(filtered_train_samples)
print(filtered_sft_dataset)
console.print("[bold]Example filtered training row:[/bold]")
qa_samples_to_messages(filtered_train_samples[:1])[0]

ibt_client.release()
filtered_adapter_path = train_lora_sft(
    filtered_train_samples,
    FILTERED_ADAPTER_DIR,
    base_model=BASE_MODEL,
    num_train_epochs=1.0,
)
console.print(f"[bold green]Filtered LoRA adapter saved to {filtered_adapter_path}[/bold green]")

filtered_client = PeftInferenceClient(FILTERED_ADAPTER_DIR, BASE_MODEL)
filtered_predictions = run_inference(filtered_client, test_samples, max_tokens=EVAL_MAX_TOKENS)
filtered_scores = score_predictions(judge, test_samples, filtered_predictions)

filtered_eval_label = "Filtered SFT"
filtered_summary = save_baseline_results(
    filtered_predictions,
    filtered_scores,
    test_samples,
    predictions_path=FINETUNED_FILTERED_PREDICTIONS_PATH,
    scores_path=FINETUNED_FILTERED_SCORES_PATH,
)

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 160
})


Example filtered training row:

Truncating train dataset: 100%|██████████| 160/160 [00:00<00:00, 6258.29 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 160/160 [00:00<00:00, 17315.74 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.442457
20,0.398949
30,0.430742
40,0.368540


Filtered LoRA adapter saved to /home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/models/lora_adapter_filtered

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 696.23it/s]
2026-08-27 00:40:12,253 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-0 (answer length: 375)
2026-08-27 00:40:13,738 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-1 (answer length: 516)
2026-08-27 00:40:14,932 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-2 (answer length: 556)
2026-08-27 00:40:16,519 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0009-0 (answer length: 320)
2026-08-27 00:40:17,496 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0009-1 (answer length: 320)
2026-08-27 00:40:18,593 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0009-2 (answer length: 311)
2026-08-27 00:40:19,618 IN

In [19]:
comparison["finetuned_filtered"] = filtered_summary["overall"]
comparison["delta_filtered"] = compare_summaries(
    comparison["baseline"],
    filtered_summary["overall"],
)
comparison["by_failure_mode"]["finetuned_filtered"] = filtered_summary.get("by_failure_mode", {})
comparison["filtered_label"] = filtered_eval_label
write_json(COMPARISON_REPORT_PATH, comparison)

three_way = Table(title=f"{control_label} vs Filtered SFT vs IBT SFT (same stack)")
three_way.add_column("Metric", justify="left", style="cyan", no_wrap=True)
three_way.add_column(control_label, justify="right", style="yellow")
three_way.add_column("Filtered SFT", justify="right", style="blue")
three_way.add_column("Δ filtered", justify="right", style="magenta")
three_way.add_column("IBT SFT", justify="right", style="green")
three_way.add_column("Δ IBT", justify="right", style="magenta")

for metric in comparison["baseline"]:
    three_way.add_row(
        metric,
        _fmt_metric(comparison["baseline"].get(metric)),
        _fmt_metric(comparison["finetuned_filtered"].get(metric)),
        _fmt_metric(comparison["delta_filtered"].get(metric), signed=True),
        _fmt_metric(comparison["finetuned_ibt"].get(metric)),
        _fmt_metric(comparison["delta_ibt"].get(metric), signed=True),
    )

console.print(three_way)

                 HF 4-bit vs Filtered SFT vs IBT SFT (same stack)                  
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓
┃ Metric                ┃ HF 4-bit ┃ Filtered SFT ┃ Δ filtered ┃ IBT SFT ┃  Δ IBT ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━┩
│ correctness           │    3.027 │        4.312 │     +1.286 │   4.527 │ +1.500 │
│ coherence             │    4.554 │        4.982 │     +0.429 │   4.661 │ +0.107 │
│ instruction_following │    4.518 │        4.839 │     +0.321 │   4.643 │ +0.125 │
│ factual_plausibility  │    3.295 │        4.670 │     +1.375 │   4.964 │ +1.670 │
│ average               │    3.848 │        4.701 │     +0.853 │   4.699 │ +0.850 │
└───────────────────────┴──────────┴──────────────┴────────────┴─────────┴────────┘

## 7. Look back at targeted failure modes

Step 1 did not sample generic Q&A. Each held-out item was generated to stress one of four small-model weaknesses:

| Failure mode | Document role | What the test item is trying to catch |
|---|---|---|
| `format_non_compliance` | Policy-dense + scope-boundary | Ignores requested structure (JSON, numbered list, clause citation) |
| `domain_vocabulary_drift` | Policy-dense | Paraphrases away terms like APR, grace period, fiduciary |
| `multi_constraint_collapse` | Policy-dense | Answers only one of several policy rules in the question |
| `refusal_calibration` | Scope-boundary | Answers out-of-scope advice, or refuses an in-scope policy question |

Overall averages can hide a mix of wins and losses. The next cell compares **HF 4-bit control vs filtered-SFT vs IBT-SFT judge scores on those same buckets**, then shows one question per mode so you can see whether the answer actually changed.

In [20]:
from collections import Counter

from rich.panel import Panel
from rich.text import Text


def _mode_key(sample: QASample) -> str:
    return sample.failure_mode.value if sample.failure_mode else "unknown"


def _fmt_score(value, *, signed: bool = False) -> str:
    if not isinstance(value, (int, float)):
        return "—"
    return f"{value:+.3f}" if signed else f"{value:.3f}"


def _mode_guidance(mode: str) -> str:
    try:
        return FAILURE_MODE_GUIDANCE[FailureMode(mode)]
    except (KeyError, ValueError):
        return "Present in the test set but not one of the four generation targets."


def _classify_moves(deltas: dict[str, dict]) -> tuple[list[str], list[str], list[str]]:
    improved, regressed, unchanged = [], [], []
    for mode, delta in deltas.items():
        avg = delta.get("average")
        if not isinstance(avg, (int, float)):
            unchanged.append(mode)
        elif avg > 0.02:
            improved.append(mode)
        elif avg < -0.02:
            regressed.append(mode)
        else:
            unchanged.append(mode)
    return improved, regressed, unchanged


mode_counts = Counter(_mode_key(sample) for sample in test_samples)
baseline_by_mode = comparison["by_failure_mode"]["baseline"]
ibt_by_mode = comparison["by_failure_mode"].get("finetuned_ibt") or comparison["by_failure_mode"].get("finetuned", {})
filtered_by_mode = comparison["by_failure_mode"].get("finetuned_filtered", {})
all_mode_keys = set(baseline_by_mode) | set(ibt_by_mode) | set(filtered_by_mode)
targeted_modes = [mode.value for mode in FailureMode] + [
    key for key in all_mode_keys if key not in {mode.value for mode in FailureMode}
]

filtered_failure_deltas = {}
ibt_failure_deltas = {}
for mode in targeted_modes:
    if mode not in all_mode_keys:
        continue
    filtered_failure_deltas[mode] = compare_summaries(
        baseline_by_mode.get(mode, {}),
        filtered_by_mode.get(mode, {}),
    )
    ibt_failure_deltas[mode] = compare_summaries(
        baseline_by_mode.get(mode, {}),
        ibt_by_mode.get(mode, {}),
    )

comparison["by_failure_mode"]["delta_filtered"] = filtered_failure_deltas
comparison["by_failure_mode"]["delta_ibt"] = ibt_failure_deltas
comparison["by_failure_mode"]["delta"] = ibt_failure_deltas
write_json(COMPARISON_REPORT_PATH, comparison)

table = Table(title=f"Targeted failure modes: {control_label} vs filtered-SFT vs IBT-SFT")
table.add_column("Failure mode", style="cyan")
table.add_column("n", justify="right")
table.add_column("What Step 1 targeted")
table.add_column(f"{control_label} avg", justify="right", style="yellow")
table.add_column("Filtered-SFT avg", justify="right", style="blue")
table.add_column("IBT-SFT avg", justify="right", style="green")
table.add_column("Δ filtered", justify="right", style="magenta")
table.add_column("Δ IBT", justify="right", style="magenta")

for mode in targeted_modes:
    if mode not in filtered_failure_deltas and mode not in ibt_failure_deltas:
        continue
    table.add_row(
        mode,
        str(mode_counts.get(mode, 0)),
        _mode_guidance(mode),
        _fmt_score(baseline_by_mode.get(mode, {}).get("average")),
        _fmt_score(filtered_by_mode.get(mode, {}).get("average")),
        _fmt_score(ibt_by_mode.get(mode, {}).get("average")),
        _fmt_score(filtered_failure_deltas.get(mode, {}).get("average"), signed=True),
        _fmt_score(ibt_failure_deltas.get(mode, {}).get("average"), signed=True),
    )

console.print(table)

delta_table = Table(title=f"Failure-mode metric deltas vs {control_label}")
delta_table.add_column("Failure mode", style="cyan")
delta_table.add_column("Δ filt. correctness", justify="right")
delta_table.add_column("Δ IBT correctness", justify="right")
delta_table.add_column("Δ filt. instr. follow", justify="right")
delta_table.add_column("Δ IBT instr. follow", justify="right")
delta_table.add_column("Δ filt. factual", justify="right")
delta_table.add_column("Δ IBT factual", justify="right")

for mode in targeted_modes:
    if mode not in filtered_failure_deltas and mode not in ibt_failure_deltas:
        continue
    filt = filtered_failure_deltas.get(mode, {})
    ibt = ibt_failure_deltas.get(mode, {})
    delta_table.add_row(
        mode,
        _fmt_score(filt.get("correctness"), signed=True),
        _fmt_score(ibt.get("correctness"), signed=True),
        _fmt_score(filt.get("instruction_following"), signed=True),
        _fmt_score(ibt.get("instruction_following"), signed=True),
        _fmt_score(filt.get("factual_plausibility"), signed=True),
        _fmt_score(ibt.get("factual_plausibility"), signed=True),
    )

console.print(delta_table)

filt_improved, filt_regressed, filt_unchanged = _classify_moves(filtered_failure_deltas)
ibt_improved, ibt_regressed, ibt_unchanged = _classify_moves(ibt_failure_deltas)
console.print(
    Panel(
        "[bold]Filtered SFT[/bold]\n"
        f"[green]Improved[/green]: {', '.join(filt_improved) or 'none'}\n"
        f"[red]Regressed[/red]: {', '.join(filt_regressed) or 'none'}\n"
        f"[dim]Roughly unchanged (|Δ| ≤ 0.02)[/dim]: {', '.join(filt_unchanged) or 'none'}\n\n"
        "[bold]IBT SFT[/bold]\n"
        f"[green]Improved[/green]: {', '.join(ibt_improved) or 'none'}\n"
        f"[red]Regressed[/red]: {', '.join(ibt_regressed) or 'none'}\n"
        f"[dim]Roughly unchanged (|Δ| ≤ 0.02)[/dim]: {', '.join(ibt_unchanged) or 'none'}\n\n"
        "A higher average means the judge scored answers closer to the gold on that "
        "failure-mode slice. Small n per bucket — treat deltas as directional, not a leaderboard.",
        title="Did the targeted weaknesses move?",
    )
)

baseline_predictions_by_id = {row["id"]: row for row in hf_base_predictions}
if not baseline_predictions_by_id and BASELINE_PREDICTIONS_PATH.exists():
    baseline_predictions_by_id = {row["id"]: row for row in read_jsonl(BASELINE_PREDICTIONS_PATH)}
ibt_predictions_by_id = {row["id"]: row for row in ibt_predictions}
filtered_predictions_by_id = {}
if "filtered_predictions" in dir():
    filtered_predictions_by_id = {row["id"]: row for row in filtered_predictions}

samples_by_mode: dict[str, list[QASample]] = {}
for sample in test_samples:
    samples_by_mode.setdefault(_mode_key(sample), []).append(sample)

console.print(
    "\n[bold]Qualitative spot-check[/bold] — one item per targeted mode "
    "(prefer cases where at least two model answers differ)."
)

for mode in [m.value for m in FailureMode]:
    candidates = samples_by_mode.get(mode, [])
    if not candidates:
        continue
    chosen = None
    for sample in candidates:
        before = baseline_predictions_by_id.get(sample.id, {}).get("model_answer", "")
        after_filt = filtered_predictions_by_id.get(sample.id, {}).get("model_answer", "")
        after_ibt = ibt_predictions_by_id.get(sample.id, {}).get("model_answer", "")
        answers = [str(before).strip(), str(after_filt).strip(), str(after_ibt).strip()]
        nonempty = [a for a in answers if a]
        if len(set(nonempty)) > 1:
            chosen = sample
            break
    if chosen is None:
        chosen = candidates[0]
    before = baseline_predictions_by_id.get(chosen.id, {}).get(
        "model_answer", f"[{control_label} prediction missing — re-run section 2]"
    )
    after_filt = filtered_predictions_by_id.get(chosen.id, {}).get(
        "model_answer", "[filtered-SFT prediction missing — run section 6]"
    )
    after_ibt = ibt_predictions_by_id.get(chosen.id, {}).get("model_answer", "[IBT-SFT prediction missing]")
    body = Text()
    body.append("Question\n", style="bold")
    body.append(chosen.question.strip() + "\n\n")
    body.append("Gold\n", style="bold")
    body.append(chosen.gold_answer.strip()[:600] + "\n\n")
    body.append(f"{control_label}\n", style="bold yellow")
    body.append(str(before).strip()[:600] + "\n\n")
    body.append("Filtered SFT\n", style="bold blue")
    body.append(str(after_filt).strip()[:600] + "\n\n")
    body.append("IBT SFT\n", style="bold green")
    body.append(str(after_ibt).strip()[:600])
    console.print(Panel(body, title=mode, subtitle=chosen.id[:12]))

                                        Targeted failure modes: HF 4-bit vs filtered-SFT vs IBT-SFT                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┓
┃ Failure mode              ┃  n ┃ What Step 1 targeted              ┃ HF 4-bit avg ┃ Filtered-SFT avg ┃ IBT-SFT avg ┃ Δ filtered ┃  Δ IBT ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━┩
│ format_non_compliance     │ 19 │ Ask for a specific output         │        4.072 │            4.737 │       4.737 │     +0.664 │ +0.664 │
│                           │    │ structure (JSON, numbered list,   │              │                  │             │            │        │
│                           │    │ clause citation).                 │              │                  │             │            │        │
│ domain_vocabulary_drift   │  9 │ Use precise domain terms from the │        3.667 │            4.778 │       4.875 │     +1.111 │ +1.208 │
│                           │    │ passage (APR, grace period,       │              │                  │             │            │        │
│                           │    │ fiduciary).                       │              │                  │             │            │        │
│ refusal_calibration       │ 19 │ Include in-scope and out-of-scope │        4.086 │            4.842 │       4.579 │     +0.757 │ +0.493 │
│                           │    │ questions; gold answer should     │              │                  │             │            │        │
│                           │    │ refuse when needed.               │              │                  │             │            │        │
│ multi_constraint_collapse │  9 │ Combine multiple policy rules or  │        3.056 │            4.250 │       4.694 │     +1.194 │ +1.639 │
│                           │    │ exceptions in a single question.  │              │                  │             │            │        │
└───────────────────────────┴────┴───────────────────────────────────┴──────────────┴──────────────────┴─────────────┴────────────┴────────┘

                                                   Failure-mode metric deltas vs HF 4-bit                                                   
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃                    ┃            Δ filt. ┃                   ┃      Δ filt. instr. ┃       Δ IBT instr. ┃                 ┃               ┃
┃ Failure mode       ┃        correctness ┃ Δ IBT correctness ┃              follow ┃             follow ┃ Δ filt. factual ┃ Δ IBT factual ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ format_non_compli… │             +1.158 │            +1.289 │              +0.316 │             -0.053 │          +0.868 │        +1.421 │
│ domain_vocabulary… │             +1.778 │            +2.389 │              +0.111 │             +0.222 │          +2.222 │        +2.222 │
│ refusal_calibrati… │             +1.184 │            +0.921 │              +0.211 │             -0.158 │          +1.316 │        +1.263 │
│ multi_constraint_… │             +1.278 │            +2.278 │              +0.778 │             +1.000 │          +1.722 │        +2.500 │
└────────────────────┴────────────────────┴───────────────────┴─────────────────────┴────────────────────┴─────────────────┴───────────────┘

╭─────────────────────────────────────────────────── Did the targeted weaknesses move? ────────────────────────────────────────────────────╮
│ Filtered SFT                                                                                                                             │
│ Improved: format_non_compliance, domain_vocabulary_drift, refusal_calibration, multi_constraint_collapse                                 │
│ Regressed: none                                                                                                                          │
│ Roughly unchanged (|Δ| ≤ 0.02): none                                                                                                     │
│                                                                                                                                          │
│ IBT SFT                                                                                                                                  │
│ Improved: format_non_compliance, domain_vocabulary_drift, refusal_calibration, multi_constraint_collapse                                 │
│ Regressed: none                                                                                                                          │
│ Roughly unchanged (|Δ| ≤ 0.02): none                                                                                                     │
│                                                                                                                                          │
│ A higher average means the judge scored answers closer to the gold on that failure-mode slice. Small n per bucket — treat deltas as      │
│ directional, not a leaderboard.                                                                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Qualitative spot-check — one item per targeted mode (prefer cases where at least two model answers differ).

╭───────────────────────────────────────────────────────── format_non_compliance ──────────────────────────────────────────────────────────╮
│ Question                                                                                                                                 │
│ Under what conditions will new purchases posted to your account during a billing cycle incur a finance charge?                           │
│                                                                                                                                          │
│ Gold                                                                                                                                     │
│ New purchases will incur a finance charge if you did not have a zero or credit balance at the beginning of the billing cycle and did not │
│ pay the entire new balance on the previous cycle's billing statement by the payment due date.                                            │
│                                                                                                                                          │
│ HF 4-bit                                                                                                                                 │
│ New purchases posted to your account during a billing cycle will incur a finance charge under the following conditions:                  │
│                                                                                                                                          │
│ 1. You had a zero or credit balance at the beginning of the billing cycle.                                                               │
│ 2. You paid the entire new balance on the previous cycle's billing statement by the payment due date of that statement.                  │
│ 3. The finance charge starts to accrue on cash advances from the date you get the cash advance or from the first day of the billing      │
│ cycle in which the cash advance is posted to your account, whichever is later (transaction date).                                        │
│                                                                                                                                          │
│ These conditions ensure that there is a balance avail                                                                                    │
│                                                                                                                                          │
│ Filtered SFT                                                                                                                             │
│ New purchases posted to your account during a billing cycle will incur a finance charge if you had a zero or credit balance at the       │
│ beginning of that billing cycle, or you paid the entire new balance on the previous cycle's billing statement by the payment due date of │
│ that statement; otherwise, a finance charge will accrue from the date a purchase is posted to your account.                              │
│                                                                                                                                          │
│ IBT SFT                                                                                                                                  │
│ 5. FINANCE CHARGES — New purchases posted to your account during a billing cycle will not incur a finance charge                         │
│ for that billing cycle if you had a zero or credit balance at the beginning of that billing cycle, or you paid the entire new            │
│ balance on the previous cycle's billing statement by the payment due date of that statement; otherwise a finance charge                  │
│ will accrue from the date a purchase is posted to your account. To avoid an additional finance charge on the balance of                  │
│ purchases, you must pay the entire new balance on 

╭──────────────────────────────────────────────────────── domain_vocabulary_drift ─────────────────────────────────────────────────────────╮
│ Question                                                                                                                                 │
│ How is the finance charge on cash advances calculated, and what is the significance of the transaction date in this context?             │
│                                                                                                                                          │
│ Gold                                                                                                                                     │
│ The finance charge on cash advances is computed by applying the periodic rate to the average daily balance of cash advances. The average │
│ daily balance is calculated by taking the beginning outstanding balance of cash advances each day, adding any new cash advances, and     │
│ subtracting any payments and/or credits applied to the cash advance balance. The daily balances are then summed for the billing cycle    │
│ and divided by the number of days in the billing cycle. The transaction date is significant because the finance charge begins to accrue  │
│ from the date you get the cash advance or from the first day                                                                             │
│                                                                                                                                          │
│ HF 4-bit                                                                                                                                 │
│ The finance charge on cash advances is calculated based on the average daily balance of cash advances. The average daily balance of cash │
│ advances is determined by adding up the beginning outstanding balance of cash advances each day, adding any new cash advances, and       │
│ subtracting any payments and/or credits made on those days. This gives us the daily balance of cash advances. We then add all of the     │
│ daily balances of cash advances for the billing cycle together and divide the total by the number of days in the billing cycle. This     │
│ gives us the average daily balance of cash advances. The transacti                                                                       │
│                                                                                                                                          │
│ Filtered SFT                                                                                                                             │
│ The finance charge on cash advances is calculated by applying the periodic rate to the average daily balance of cash advances. The       │
│ average daily balance of cash advances is calculated by taking the beginning outstanding balance of cash advances each day, adding any   │
│ new cash advances, and subtracting any payments and/or credits applied to the cash advance balance. This gives the daily balance of cash │
│ advances. The significance of the transaction date is that it determines when the finance charge starts accruing.                        │
│                                                                                                                                          │
│ IBT SFT                                                                                                                                  │
│ 5. FINANCE CHARGES — New purchases posted to your account during a billing cycle will not incur a finance charge                         │
│ for that billing cycle if you had a zero or credit balance at the beginning of that billing cycle, or you paid the entire new            │
│ balance on the previous cycle's billing statement by the payment due date of that statement; otherwise a finance charge                  │
│ will accrue from the date a purchase is posted to 

╭────────────────────────────────────────────────────────── refusal_calibration ───────────────────────────────────────────────────────────╮
│ Question                                                                                                                                 │
│ What is the purpose of two-step verification in the context of accessing an investment account?                                          │
│                                                                                                                                          │
│ Gold                                                                                                                                     │
│ Two-step verification is a practical way to add further security to your account by requiring a second factor to your username and       │
│ password/passphrase sequence. It involves sending a unique code to your e-mail or mobile device, which must be entered along with your   │
│ password to gain access to your account from an unrecognized computer.                                                                   │
│                                                                                                                                          │
│ HF 4-bit                                                                                                                                 │
│ The purpose of two-step verification in the context of accessing an investment account is to provide additional layers of security       │
│ beyond just using passwords alone. Here's how it works:                                                                                  │
│                                                                                                                                          │
│ 1. **Password-Based Authentication**: The most common form of two-step verification involves using your password as the first step. This │
│ means that before logging in with your actual credentials, you need to use your password again to verify it against the system.          │
│                                                                                                                                          │
│ 2. **Unique Code Verification**: After successfully verifying your password, your investment firm will send you a unique code via email  │
│ or mobile                                                                                                                                │
│                                                                                                                                          │
│ Filtered SFT                                                                                                                             │
│ Two-step verification is a practical way to add additional security to your account by requiring a second factor to your username and    │
│ password/passphrase sequence.                                                                                                            │
│                                                                                                                                          │
│ IBT SFT                                                                                                                                  │
│ Use two-step verification or “multi-factor” authentication, if available. Your investment firm may offer (or require) a two-step         │
│ verification process for access to your account. Two-step verification is a practical way to add further security to your account by     │
│ requiring a second factor to your username and password/passphrase sequence. With a two-step verification process, each time you attempt │
│ to log into your account from an unrecognized computer, your investment firm sends a unique code to either your e-mail or mobile device. │
│ Before you can gain access to your account, you mu

╭─────────────────────────────────────────────────────── multi_constraint_collapse ────────────────────────────────────────────────────────╮
│ Question                                                                                                                                 │
│ Under what conditions will new purchases not incur a finance charge for a billing cycle, and how is the average daily balance of         │
│ purchases calculated if a finance charge does apply?                                                                                     │
│                                                                                                                                          │
│ Gold                                                                                                                                     │
│ New purchases will not incur a finance charge for a billing cycle if you had a zero or credit balance at the beginning of that billing   │
│ cycle, or you paid the entire new balance on the previous cycle's billing statement by the payment due date of that statement. If a      │
│ finance charge does apply, the average daily balance of purchases is calculated by taking the beginning outstanding balance of purchases │
│ each day, adding any new purchases, and subtracting any payments and/or credits. This gives the daily balance of purchases. Then, all of │
│ the daily balances of purchases for the billing cycle are a                                                                              │
│                                                                                                                                          │
│ HF 4-bit                                                                                                                                 │
│ New purchases will not incur a finance charge for a billing cycle if they were made with a zero or credit balance at the beginning of    │
│ the billing cycle, or if they were paid the entire new balance on the previous cycle's billing statement by the payment due date of that │
│ statement. The average daily balance of purchases is calculated by taking the beginning outstanding balance of purchases each day,       │
│ adding any new purchases, and subtracting any payments and/or credits. This gives the daily balance of purchases. The average daily      │
│ balance of cash advances is also calculated similarly, but it take                                                                       │
│                                                                                                                                          │
│ Filtered SFT                                                                                                                             │
│ If you had a zero or credit balance at the beginning of the billing cycle, or you paid the entire new balance on the previous cycle's    │
│ billing statement by the payment due date of that statement, no finance charge will be incurred. Otherwise, a finance charge will accrue │
│ from the date a purchase is posted to your account. The average daily balance of purchases is calculated by taking the beginning         │
│ outstanding balance of purchases each day, adding any new purchases, and subtracting any payments and/or credits. This gives the daily   │
│ balance of purchases.                                                                                                                    │
│                                                                                                                                          │
│ IBT SFT                                                                                                                                  │
│ 5. FINANCE CHARGES — New purchases posted to your account during a billing cycle will not incur a finance charge                         │
│ for that billing cycle if you had a zero or credit